<a href="https://colab.research.google.com/github/baanujan-18/Statistical-Learning-e20030/blob/main/Assignment_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kalman Filter Assignment

This notebook contains:

1. Analytical derivation of the linear Kalman filter  
2. A one-dimensional scalar example  
3. A two-dimensional constant-velocity GPS position estimator  
4. Simulation, visualization, and an optional real-data workflow  

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

np.set_printoptions(precision=4, suppress=True)

# 1. Analytical Derivation

Consider the linear-Gaussian state-space model

$$
x_k^- = A_{k-1}x_{k-1}^+ + G_{k-1}w_{k-1},
$$

$$
y_k^- = H_kx_k^- + z_k,
$$

where

$$
x_{k-1}^+ \sim \mathcal{N}(m_{k-1},P_{k-1}),
$$

$$
w_{k-1} \sim \mathcal{N}(0,\Sigma_p),
\qquad
z_k \sim \mathcal{N}(0,\Sigma_m).
$$

The state, process noise, and measurement noise are assumed to be mutually independent.

## 1.1 Predicted State Distribution

The predicted state is

$$
x_k^- = A_{k-1}x_{k-1}^+ + G_{k-1}w_{k-1}.
$$

Because the previous state and the process noise are Gaussian, the predicted state is also Gaussian:

$$
x_k^- \sim \mathcal{N}(m_k^-,P_k^-).
$$

The predicted mean is

$$
m_k^- = \mathbb{E}[x_k^-].
$$

Substituting the state equation,

$$
m_k^- =
\mathbb{E}
\left[
A_{k-1}x_{k-1}^+
+
G_{k-1}w_{k-1}
\right].
$$

Using the linearity of expectation,

$$
m_k^- =
A_{k-1}\mathbb{E}[x_{k-1}^+]
+
G_{k-1}\mathbb{E}[w_{k-1}].
$$

Since

$$
\mathbb{E}[x_{k-1}^+] = m_{k-1}
$$

and

$$
\mathbb{E}[w_{k-1}] = 0,
$$

we obtain

$$
\boxed{
m_k^- = A_{k-1}m_{k-1}
}.
$$

The predicted covariance is

$$
P_k^- = \operatorname{Var}(x_k^-).
$$

Therefore,

$$
P_k^- =
\operatorname{Var}
\left(
A_{k-1}x_{k-1}^+
+
G_{k-1}w_{k-1}
\right).
$$

Because the previous state and the process noise are independent,

$$
P_k^- =
A_{k-1}P_{k-1}A_{k-1}^{T}
+
G_{k-1}\Sigma_pG_{k-1}^{T}.
$$

Thus,

$$
\boxed{
x_k^- \sim
\mathcal{N}
\left(
A_{k-1}m_{k-1},
A_{k-1}P_{k-1}A_{k-1}^{T}
+
G_{k-1}\Sigma_pG_{k-1}^{T}
\right)
}.
$$

## 1.2 Predicted Measurement Distribution

The predicted measurement is

$$
y_k^- = H_kx_k^- + z_k.
$$

Since the predicted state is Gaussian,

$$
x_k^- \sim \mathcal{N}(m_k^-,P_k^-),
$$

and the measurement noise is Gaussian,

$$
z_k \sim \mathcal{N}(0,\Sigma_m),
$$

the predicted measurement is also Gaussian.

The predicted measurement mean is

$$
\mathbb{E}[y_k^-]
=
\mathbb{E}[H_kx_k^- + z_k].
$$

Using the linearity of expectation,

$$
\mathbb{E}[y_k^-]
=
H_k\mathbb{E}[x_k^-]
+
\mathbb{E}[z_k].
$$

Since

$$
\mathbb{E}[x_k^-]=m_k^-
$$

and

$$
\mathbb{E}[z_k]=0,
$$

we obtain

$$
\boxed{
\mathbb{E}[y_k^-]=H_km_k^-
}.
$$

The predicted measurement covariance is

$$
\operatorname{Var}(y_k^-)
=
\operatorname{Var}(H_kx_k^-+z_k).
$$

Because the predicted state and the measurement noise are independent,

$$
\operatorname{Var}(y_k^-)
=
H_kP_k^-H_k^T+\Sigma_m.
$$

Therefore,

$$
\boxed{
y_k^- \sim
\mathcal{N}
\left(
H_km_k^-,
H_kP_k^-H_k^T+\Sigma_m
\right)
}.
$$

## 1.3 Joint State–Measurement Distribution

We now find the joint distribution of the predicted state and the predicted measurement.

The predicted state is

$$
x_k^- \sim \mathcal{N}(m_k^-,P_k^-),
$$

and the predicted measurement is

$$
y_k^- = H_kx_k^- + z_k.
$$

The mean of the joint random vector is

$$
\mathbb{E}
\left[
\begin{bmatrix}
x_k^- \\
y_k^-
\end{bmatrix}
\right]
=
\begin{bmatrix}
\mathbb{E}[x_k^-] \\
\mathbb{E}[y_k^-]
\end{bmatrix}.
$$

Therefore,

$$
\boxed{
\mathbb{E}
\left[
\begin{bmatrix}
x_k^- \\
y_k^-
\end{bmatrix}
\right]
=
\begin{bmatrix}
m_k^- \\
H_km_k^-
\end{bmatrix}
}.
$$

The cross-covariance between the predicted state and the predicted measurement is

$$
\operatorname{Cov}(x_k^-,y_k^-)
=
\operatorname{Cov}
\left(
x_k^-,
H_kx_k^-+z_k
\right).
$$

Since the predicted state and the measurement noise are independent,

$$
\operatorname{Cov}(x_k^-,z_k)=0.
$$

Thus,

$$
\operatorname{Cov}(x_k^-,y_k^-)
=
\operatorname{Cov}(x_k^-,H_kx_k^-).
$$

Therefore,

$$
\boxed{
\operatorname{Cov}(x_k^-,y_k^-)
=
P_k^-H_k^T
}.
$$

Similarly,

$$
\boxed{
\operatorname{Cov}(y_k^-,x_k^-)
=
H_kP_k^-
}.
$$

The covariance of the predicted measurement is

$$
\boxed{
\operatorname{Var}(y_k^-)
=
H_kP_k^-H_k^T+\Sigma_m
}.
$$

Therefore, the joint distribution is

$$
\boxed{
\begin{bmatrix}
x_k^- \\
y_k^-
\end{bmatrix}
\sim
\mathcal{N}
\left(
\begin{bmatrix}
m_k^- \\
H_km_k^-
\end{bmatrix},
\begin{bmatrix}
P_k^- & P_k^-H_k^T \\
H_kP_k^- & H_kP_k^-H_k^T+\Sigma_m
\end{bmatrix}
\right)
}.
$$

## 1.4 Measurement Update and Conditional Distribution

At time step \(k\), the measurement random variable \(y_k^-\) takes the observed numerical value

$$
y_k^- = y_k^{\mathrm{obs}}.
$$

The posterior state is defined as the conditional random variable

$$
x_k^+
=
\left(
x_k^- \mid y_k^- = y_k^{\mathrm{obs}}
\right).
$$

Since the joint distribution of \(x_k^-\) and \(y_k^-\) is Gaussian, the conditional distribution is also Gaussian:

$$
\boxed{
x_k^+ \sim \mathcal{N}(m_k,P_k)
}.
$$

Define the innovation covariance as

$$
S_k
=
H_kP_k^-H_k^T+\Sigma_m.
$$

The Kalman gain is

$$
\boxed{
K_k
=
P_k^-H_k^TS_k^{-1}
}.
$$

Equivalently,

$$
\boxed{
K_k
=
P_k^-H_k^T
\left(
H_kP_k^-H_k^T+\Sigma_m
\right)^{-1}
}.
$$

The innovation, or measurement residual, is

$$
\boxed{
v_k
=
y_k^{\mathrm{obs}}-H_km_k^-
}.
$$

The updated mean is

$$
m_k
=
m_k^-
+
P_k^-H_k^TS_k^{-1}
\left(
y_k^{\mathrm{obs}}-H_km_k^-
\right).
$$

Using the Kalman gain, this becomes

$$
\boxed{
m_k
=
m_k^-+K_k
\left(
y_k^{\mathrm{obs}}-H_km_k^-
\right)
}.
$$

The updated covariance is

$$
P_k
=
P_k^-
-
P_k^-H_k^TS_k^{-1}H_kP_k^-.
$$

Using the Kalman gain,

$$
\boxed{
P_k
=
\left(
I-K_kH_k
\right)P_k^-
}.
$$

Therefore, the posterior distribution is

$$
\boxed{
x_k^+
\sim
\mathcal{N}
\left(
m_k^-+K_k
\left(
y_k^{\mathrm{obs}}-H_km_k^-
\right),
\left(
I-K_kH_k
\right)P_k^-
\right)
}.
$$

## 1.5 Conditional Expectation and Conditional Variance

From the conditional Gaussian distribution,

$$
x_k^+
=
\left(
x_k^- \mid y_k^- = y_k^{\mathrm{obs}}
\right)
\sim
\mathcal{N}(m_k,P_k).
$$

Therefore, the conditional expectation is

$$
\boxed{
\mathbb{E}
\left[
x_k^-
\mid
y_k^- = y_k^{\mathrm{obs}}
\right]
=
m_k
}.
$$

Using the Kalman filter update equation,

$$
\boxed{
\mathbb{E}
\left[
x_k^-
\mid
y_k^- = y_k^{\mathrm{obs}}
\right]
=
m_k^-
+
P_k^-H_k^T
\left(
H_kP_k^-H_k^T+\Sigma_m
\right)^{-1}
\left(
y_k^{\mathrm{obs}}-H_km_k^-
\right)
}.
$$

The conditional variance is

$$
\boxed{
\operatorname{Var}
\left(
x_k^-
\mid
y_k^- = y_k^{\mathrm{obs}}
\right)
=
P_k
}.
$$

Using the covariance update formula,

$$
\boxed{
\operatorname{Var}
\left(
x_k^-
\mid
y_k^- = y_k^{\mathrm{obs}}
\right)
=
P_k^-
-
P_k^-H_k^T
\left(
H_kP_k^-H_k^T+\Sigma_m
\right)^{-1}
H_kP_k^-
}.
$$

Equivalently,

$$
\boxed{
\operatorname{Var}
\left(
x_k^-
\mid
y_k^- = y_k^{\mathrm{obs}}
\right)
=
\left(
I-K_kH_k
\right)P_k^-
}.
$$